In [25]:
import pandas as pd
import requests
import tqdm
import time



In [ ]:

API_KEY = ""

address = "הרצל 1, תל אביב, ישראל"
url = "https://maps.googleapis.com/maps/api/geocode/json"

params = {
    "address": address,
    "key": API_KEY
}

res = requests.get(url, params=params).json()

location = res["results"][0]["geometry"]["location"]
print(location["lat"], location["lng"])

32.0634795 34.7700272


In [23]:
df_kalpi_addresses_with_coordinates = pd.read_csv("../data/kalpi_address_with_coords.csv")

In [24]:
df_kalpi_addresses_with_coordinates.at[0, "coordinates"]

'(31.8033163, 35.2167565)'

In [28]:
count_missing_coordinates = 0
coordinates_cache = {}
for index, row in tqdm.tqdm(df_kalpi_addresses_with_coordinates.iterrows(), total=df_kalpi_addresses_with_coordinates.shape[0]):
    if pd.isna(row["coordinates"]):
        address = f"{row['kalpi_address']}, {row['locality_name']}, ישראל"
        if address in coordinates_cache:
            df_kalpi_addresses_with_coordinates.at[index, "coordinates"] = coordinates_cache[address]
            continue
        
        params = {
            "address": address,
            "key": API_KEY
        }

        res = requests.get(url, params=params).json()
        time.sleep(0.05)
        if res["results"]:
            location = res["results"][0]["geometry"]["location"]
            location_str = f"({location['lat']},{location['lng']})"
            coordinates_cache[address] = location_str
            df_kalpi_addresses_with_coordinates.at[index, "coordinates"] = location_str


100%|██████████| 11547/11547 [02:22<00:00, 81.20it/s] 


In [32]:
df_kalpi_addresses_with_coordinates.to_csv("../data/kalpi_address_with_coords.csv", index=False)
